Run this notebook with the `.venv-tts` kernel (separate venv — chatterbox-tts pins numpy>=2.0 / torch==2.6.0, which conflict with the main project's onnxruntime pin).

In [1]:
import json
import os

import torchaudio as ta
from chatterbox.tts import ChatterboxTTS
from huggingface_hub import hf_hub_download

ImportError: cannot import name 'Tensor' from partially initialized module 'torch' (most likely due to a circular import) (/Users/michaeleko/Documents/Projects/challenge-audio/ordo-ai/.venv-tts/lib/python3.11/site-packages/torch/__init__.py)

In [ ]:
DATA_PATH = os.path.join("..", "data", "normalized", "intent_dataset_normalized.jsonl")
AUDIO_DIR = os.path.join("..", "data", "audio")
MANIFEST_PATH = os.path.join("..", "data", "intent_dataset_normalized.jsonl")

REPO_ID = "grandhigh/Chatterbox-TTS-Indonesian"
CKPT_FILES = ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "tokenizer.json"]

VOICE_PROMPTS = {
    "formal": "example1.wav",
    "informal": "example2.wav",
}

DEVICE = "mps"
LIMIT = None  # set to None to run the full dataset

os.makedirs(AUDIO_DIR, exist_ok=True)

In [ ]:
ckpt_dir = None
for fpath in CKPT_FILES:
    local_path = hf_hub_download(repo_id=REPO_ID, filename=fpath)
    ckpt_dir = os.path.dirname(local_path)

voice_prompt_paths = {
    style: hf_hub_download(repo_id=REPO_ID, filename=fname)
    for style, fname in VOICE_PROMPTS.items()
}

model = ChatterboxTTS.from_local(ckpt_dir, device=DEVICE)
print(f"loaded {REPO_ID} on {DEVICE}")

/Users/michaeleko/Documents/Projects/challenge-audio/ordo-ai/.venv-tts/lib/python3.11/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000
loaded grandhigh/Chatterbox-TTS-Indonesian on mps


In [ ]:
records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

if LIMIT is not None:
    records = records[:LIMIT]

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 1100 records from ../data/synthetic_id_formal_informal_normalized.jsonl


In [ ]:
manifest = []
for i, r in enumerate(records):
    prompt_path = voice_prompt_paths[r["style"]]
    filename = f"{r['id']:04d}_{r['style']}.wav"
    out_path = os.path.join(AUDIO_DIR, filename)

    wav = model.generate(r["text_normalized"], audio_prompt_path=prompt_path)
    ta.save(out_path, wav, model.sr)

    manifest.append(
        {
            "id": r["id"],
            "audio_path": os.path.join("audio", filename),
            "text": r["text"],
            "text_normalized": r["text_normalized"],
            "style": r["style"],
            "domain": r["domain"],
            "voice_prompt": VOICE_PROMPTS[r["style"]],
        }
    )
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(records)} done")

print(f"generated {len(manifest)} audio files in {AUDIO_DIR}")

/Users/michaeleko/.pyenv/versions/3.11.13/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
`sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.
Sampling:   8%|▊         | 78/1000 [00:02<00:30, 29.88it/s]


10/1100 done


Sampling:   6%|▌         | 62/1000 [00:02<00:37, 24.79it/s]


20/1100 done


Sampling:   8%|▊         | 77/1000 [00:02<00:32, 28.03it/s]


...

1080/1100 done


Sampling:   8%|▊         | 85/1000 [00:02<00:32, 28.55it/s]


1090/1100 done


Sampling:  13%|█▎        | 126/1000 [00:04<00:33, 26.10it/s]


1100/1100 done
generated 1100 audio files in ../data/audio


In [ ]:
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for r in manifest:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote manifest with {len(manifest)} rows to {MANIFEST_PATH}")

wrote manifest with 1100 rows to ../data/synthetic_id_formal_informal_manifest.jsonl
